In [ ]:
#### Get highest pickscore and analysis win_rates  ####
import os
import pandas as pd
import matplotlib.pyplot as plt

# all_information_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real_images_uid_prompt.csv"
base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all"
# base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"
# specific_csv_file_path = os.path.join(base_dir, "top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all-uids.csv")
pickscore_csv_file_path = os.path.join(base_dir, "pickscore/pickscore_score.csv")
# imagereward_csv_file_path = os.path.join(base_dir, "imagereward/imagereward_score.csv")
# hpsv3_csv_file_path = os.path.join(base_dir, "hpsv3/hpsv3_score.csv")
# deqa_csv_file_path = os.path.join(base_dir, "deqa/deqa_score.csv")
# maagiqa_csv_file_path = os.path.join(base_dir, "ma-agiqa/ma-agiqa_score.csv")
# aesthetic_csv_file_path = os.path.join(base_dir, "aesthetic/aesthetic_score.csv")
colorfulness_csv_file_path = os.path.join(base_dir, "colorfulness/colorfulness_score.csv")
# anime_csv_file_path = os.path.join(base_dir, "qwen3_anime/qwen3_anime.csv")

# all_information_df = pd.read_csv(all_information_csv_file_path, dtype={"uid": str})
pickscore_df = pd.read_csv(pickscore_csv_file_path, dtype={"uid": str})
pickscore_gap_threshold = 0.02
valid_pickscore_uids = pickscore_df[pickscore_df['real_image_score'] - pickscore_df['fake_image_score'] > pickscore_gap_threshold]['uid'].tolist()
# imagereward_df = pd.read_csv(imagereward_csv_file_path, dtype={"uid": str})
# hpsv3_df = pd.read_csv(hpsv3_csv_file_path, dtype={"uid": str})
# deqa_df = pd.read_csv(deqa_csv_file_path, dtype={"uid": str})
# maagiqa_df = pd.read_csv(maagiqa_csv_file_path, dtype={"uid": str})
# aesthetic_df = pd.read_csv(aesthetic_csv_file_path, dtype={"uid": str})
colorfulness_df = pd.read_csv(colorfulness_csv_file_path, dtype={"uid": str})

valid_real_image_uids = valid_pickscore_uids
# anime_df = pd.read_csv(anime_csv_file_path, dtype={"uid": str})
# valid_real_image_uids = anime_df[anime_df['anime'] == 'no']['uid'].tolist()
# valid_real_image_uids = list(set(valid_real_image_uids) & set(valid_pickscore_uids))

# if specific_csv_file_path is not None:
#     specific_df = pd.read_csv(specific_csv_file_path, dtype={"uid": str})
#     valid_real_image_uids = specific_df['uid'].tolist()
    
pickscore_df = pickscore_df[pickscore_df['uid'].isin(valid_real_image_uids)]
# imagereward_df = imagereward_df[imagereward_df['uid'].isin(valid_real_image_uids)]
# hpsv3_df = hpsv3_df[hpsv3_df['uid'].isin(valid_real_image_uids)]
# deqa_df = deqa_df[deqa_df['uid'].isin(valid_real_image_uids)]
# maagiqa_df = maagiqa_df[maagiqa_df['uid'].isin(valid_real_image_uids)]
# aesthetic_df = aesthetic_df[aesthetic_df['uid'].isin(valid_real_image_uids)]
colorfulness_df = colorfulness_df[colorfulness_df['uid'].isin(valid_real_image_uids)]
#### Calculate win rate: real_image_score > fake_image_score ####
import pandas as pd

# 计算每个reward model中满足 real_image_score > fake_image_score 的比例
# 注意：相同分数按0.5计算（平局）
reward_models = {
    # 'pickscore': pickscore_df,
    # 'imagereward': imagereward_df,
    # 'hpsv3': hpsv3_df,
    # 'deqa': deqa_df,
    # 'maagiqa': maagiqa_df,
    # 'aesthetic': aesthetic_df,
    'colorfulness': colorfulness_df
}

print("=" * 80)
print("Analysis: Percentage of UIDs where real_image_score > fake_image_score")
print("Note: Ties (real == fake) count as 0.5")
print("=" * 80)
print()

results = {}
for model_name, df in reward_models.items():
    total_count = len(df)
    win_count = len(df[df['real_image_score'] > df['fake_image_score']])
    tie_count = len(df[df['real_image_score'] == df['fake_image_score']])
    lose_count = len(df[df['real_image_score'] < df['fake_image_score']])
    
    win_rate = ((win_count + tie_count * 0.5) / total_count * 100) if total_count > 0 else 0
    
    results[model_name] = {
        'total': total_count,
        'win': win_count,
        'tie': tie_count,
        'lose': lose_count,
        'win_rate': win_rate
    }
    
    print(f"{model_name:15s}: Win={win_count:6d}, Tie={tie_count:6d}, Lose={lose_count:6d}")
    print(f"{'':15s}  Win Rate = {win_rate:6.2f}% ({(win_count + tie_count * 0.5):.1f} / {total_count})")
    print()

print("=" * 80)
print("Summary Table")
print("=" * 80)
summary_df = pd.DataFrame(results).T
summary_df.columns = ['Total UIDs', 'Win', 'Tie', 'Lose', 'Win Rate (%)']
print(summary_df.to_string())

#### Function to analyze difference statistics and plot histogram for any reward model ####
def analyze_score_difference(model_name, df, bins=50, figsize=(10, 6), dpi=300, plot_hist=True):
    """
    分析 reward model 的 score difference 统计信息并绘制直方图
    
    Parameters:
    -----------
    model_name : str
        reward model 的名称
    df : pandas.DataFrame
        包含 'real_image_score' 和 'fake_image_score' 列的 DataFrame
    bins : int
        直方图的 bins 数量，默认 50
    figsize : tuple
        图像大小，默认 (10, 6)
    dpi : int
        图像分辨率，默认 300
    plot_hist : bool
        是否绘制直方图，默认 True
    """
    print("\n" + "=" * 80)
    print(f"{model_name.capitalize()} Analysis: Mean difference (real_image_score - fake_image_score)")
    print("=" * 80)
    
    # 计算每个记录的差值
    df = df.copy()
    df['score_diff'] = df['real_image_score'] - df['fake_image_score']
    
    # 计算统计信息
    mean_diff = df['score_diff'].mean()
    var_diff = df['score_diff'].var()
    std_diff = df['score_diff'].std()
    min_diff = df['score_diff'].min()
    max_diff = df['score_diff'].max()
    
    print(f"\nStatistics of difference (real_image_score - fake_image_score) for all records:")
    print(f"  Mean:     {mean_diff:.6f}")
    print(f"  Variance: {var_diff:.6f}")
    print(f"  Std:      {std_diff:.6f}")
    print(f"  Min:      {min_diff:.6f}")
    print(f"  Max:      {max_diff:.6f}")
    
    # 绘制直方图
    if plot_hist:
        plt.figure(figsize=figsize, dpi=dpi)
        plt.hist(df['score_diff'], bins=bins, edgecolor='black', alpha=0.7, color='steelblue')
        plt.axvline(0, color='red', linestyle='-', linewidth=2, label='Zero line')
        plt.xlabel(f'{model_name.capitalize()} Difference (real_image_score - fake_image_score)', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.title(f'Histogram of {model_name.capitalize()} Difference', fontsize=14, fontweight='bold')
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()
    
    return {
        'mean': mean_diff,
        'variance': var_diff,
        'std': std_diff,
        'min': min_diff,
        'max': max_diff
    }

#### Analyze colorfulness difference ####
colorfulness_stats = analyze_score_difference('colorfulness', colorfulness_df)

#### Example: Analyze other reward models ####
# Uncomment the lines below to analyze other reward models:
# pickscore_stats = analyze_score_difference('pickscore', pickscore_df)
# imagereward_stats = analyze_score_difference('imagereward', imagereward_df)
# hpsv3_stats = analyze_score_difference('hpsv3', hpsv3_df)
# deqa_stats = analyze_score_difference('deqa', deqa_df)
# maagiqa_stats = analyze_score_difference('maagiqa', maagiqa_df)
# aesthetic_stats = analyze_score_difference('aesthetic', aesthetic_df)


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

inpainting_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"
text2image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all"

colorfulness_inpainting_df = pd.read_csv(os.path.join(inpainting_dir, "colorfulness/colorfulness_score.csv"), dtype={"uid": str})
colorfulness_text2image_df = pd.read_csv(os.path.join(text2image_dir, "colorfulness/colorfulness_score.csv"), dtype={"uid": str})
inpainting_uid_list = pd.read_csv(os.path.join(inpainting_dir, "random_9984_images_no_anime_pickscore_0.02-hpdv3_all-uids.csv"), dtype={"uid": str})['uid'].tolist()
text2image_uid_list = pd.read_csv(os.path.join(text2image_dir, "random_9984_images_no_anime_pickscore_0.02-hpdv3_all-uids.csv"), dtype={"uid": str})['uid'].tolist()

colorfulness_inpainting_df = colorfulness_inpainting_df[colorfulness_inpainting_df['uid'].isin(inpainting_uid_list)]
colorfulness_text2image_df = colorfulness_text2image_df[colorfulness_text2image_df['uid'].isin(text2image_uid_list)]

#### Plot distribution of real_image_score and fake_image_score ####
# Set style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11

# Define colors
color_real = '#2E86AB'  # Modern blue
color_fake = '#F24236'  # Modern red
color_real_mean = '#1B4F72'  # Darker blue for mean line
color_fake_mean = '#C0392B'  # Darker red for mean line

# Calculate global x-axis range for consistent scaling
all_real_scores = pd.concat([
    colorfulness_inpainting_df['real_image_score'],
    colorfulness_text2image_df['real_image_score']
])
all_fake_scores = pd.concat([
    colorfulness_inpainting_df['fake_image_score'],
    colorfulness_text2image_df['fake_image_score']
])
x_min = min(all_real_scores.min(), all_fake_scores.min())
x_max = max(all_real_scores.max(), all_fake_scores.max())
# Add a small margin for better visualization
x_margin = (x_max - x_min) * 0.02
x_range = [x_min - x_margin, x_max + x_margin]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300)
fig.suptitle('Colorfulness Score Distribution: Real vs Fake Images', 
             fontsize=16, fontweight='bold', y=1.02)

# Helper function to plot distribution
def plot_distribution(ax, real_scores, fake_scores, title, dataset_name, fake_label='Fake Image', x_range=None):
    # Plot histograms with better styling
    n_bins = 60
    ax.hist(real_scores, bins=n_bins, alpha=0.65, label='Real Image', 
            color=color_real, edgecolor='white', linewidth=0.5, density=False)
    ax.hist(fake_scores, bins=n_bins, alpha=0.65, label=fake_label, 
            color=color_fake, edgecolor='white', linewidth=0.5, density=False)
    
    # Set x-axis range if provided
    if x_range is not None:
        ax.set_xlim(x_range)
    
    # Styling
    ax.set_xlabel('Colorfulness Score', fontsize=12, fontweight='medium')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='medium')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    ax.legend(loc='upper right', frameon=True, fancybox=True, shadow=True, 
              framealpha=0.95, fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')

# Plot for Inpainting
plot_distribution(axes[0], 
                  colorfulness_inpainting_df['real_image_score'],
                  colorfulness_inpainting_df['fake_image_score'],
                  '(Real, Saliency Inpainting)', 'Saliency Inpainting', fake_label='Saliency Inpainting', x_range=x_range)

# Plot for Text2Image
plot_distribution(axes[1], 
                  colorfulness_text2image_df['real_image_score'],
                  colorfulness_text2image_df['fake_image_score'],
                  '(Real, Text2Image)', 'Text2Image', fake_label='Text2Image', x_range=x_range)

plt.tight_layout()
plt.show()

# Print statistics
print("=" * 80)
print("Colorfulness Score Statistics")
print("=" * 80)

# Calculate mean differences
inpainting_diff = colorfulness_inpainting_df['real_image_score'] - colorfulness_inpainting_df['fake_image_score']
text2image_diff = colorfulness_text2image_df['real_image_score'] - colorfulness_text2image_df['fake_image_score']
inpainting_mean_diff = inpainting_diff.mean()
text2image_mean_diff = text2image_diff.mean()

print("\n(real image, inpainting image):")
print(f"  Real Image Score - Mean: {colorfulness_inpainting_df['real_image_score'].mean():.4f}, "
      f"Std: {colorfulness_inpainting_df['real_image_score'].std():.4f}, "
      f"Min: {colorfulness_inpainting_df['real_image_score'].min():.4f}, "
      f"Max: {colorfulness_inpainting_df['real_image_score'].max():.4f}")
print(f"  Fake Image Score - Mean: {colorfulness_inpainting_df['fake_image_score'].mean():.4f}, "
      f"Std: {colorfulness_inpainting_df['fake_image_score'].std():.4f}, "
      f"Min: {colorfulness_inpainting_df['fake_image_score'].min():.4f}, "
      f"Max: {colorfulness_inpainting_df['fake_image_score'].max():.4f}")
print(f"  Mean Difference (Real - Fake): {inpainting_mean_diff:.4f}")

print("\n(real image, text2image image):")
print(f"  Real Image Score - Mean: {colorfulness_text2image_df['real_image_score'].mean():.4f}, "
      f"Std: {colorfulness_text2image_df['real_image_score'].std():.4f}, "
      f"Min: {colorfulness_text2image_df['real_image_score'].min():.4f}, "
      f"Max: {colorfulness_text2image_df['real_image_score'].max():.4f}")
print(f"  Fake Image Score - Mean: {colorfulness_text2image_df['fake_image_score'].mean():.4f}, "
      f"Std: {colorfulness_text2image_df['fake_image_score'].std():.4f}, "
      f"Min: {colorfulness_text2image_df['fake_image_score'].min():.4f}, "
      f"Max: {colorfulness_text2image_df['fake_image_score'].max():.4f}")
print(f"  Mean Difference (Real - Fake): {text2image_mean_diff:.4f}")

print("\n" + "=" * 80)
print("Summary of Mean Differences")
print("=" * 80)
print(f"Inpainting Dataset - Mean Difference: {inpainting_mean_diff:.4f}")
print(f"Text2Image Dataset - Mean Difference: {text2image_mean_diff:.4f}")

